
# 00 &middot; Start here

**~20 minutes. Do this one first.** By the end you will have a (very simple) model on the
leaderboard.

### What you are working on

In October 2025, OpenADMET and Expansion Therapeutics released ~7,000
molecules from a real lead-optimisation campaign targeting RNA-mediated
diseases (myotonic dystrophy, ALS, dementia). Nine ADMET endpoints were
measured.

The split is **temporal**: you get early-stage compounds, you predict
late-stage ones.

### The plan for this notebook

1. Load the data and look at it
2. Understand the scoring metric
3. Build the null model
4. Submit it

---
### Setup

Change `CHANGE-ME` to your pair name (pick something short and memorable &mdash; it is what shows up on the leaderboard).

In [2]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec

!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/test/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="test")

pair      : test
workspace : /content/drive/MyDrive/admet_hackathon/test
prediction files on hand: 0


'/content/drive/MyDrive/admet_hackathon/test'

---
## 1. The data

Two files. `train` has molecules **and** measurements. `test` has molecules
only &mdash; the measurements are what you are predicting.

In [3]:
train = common.load_train()   # log-scale columns, ready to model
test  = common.load_test()    # blinded: endpoint columns are all NaN

print(f"train: {len(train):,} molecules")
print(f"test : {len(test):,} molecules")
train.head()

train: 5,326 molecules
test : 2,282 molecules


,Molecule Name,SMILES,LogD,LogS,Log_HLM_CLint,Log_MLM_CLint,Log_Caco_Papp_AB,Log_Caco_ER,Log_Mouse_PPB,Log_Mouse_BPB,Log_Mouse_MPB
0,E-0001321,CN1CCC[C@H]1COc1ccc(-c2nc3cc(-c4ccc5[nH]c(-c6c...,NaN,NaN,1.758912,2.263162,NaN,NaN,NaN,NaN,NaN
1,E-0001780,COc1ccc2c(c1)c1cc3cnccc3c(C)c1n2C,NaN,NaN,2.207904,3.131009,NaN,NaN,NaN,NaN,NaN
2,E-0001827,Cc1c2ccncc2cc2c3cc(OCCCN4CCN(C)CC4)ccc3n(C)c12,NaN,NaN,NaN,2.288920,NaN,NaN,NaN,NaN,NaN
3,E-0002019,CN(C)CCCOc1ccc(-c2nc3cc(NC(=O)c4ccc5[nH]c(-c6c...,NaN,NaN,1.021189,NaN,NaN,NaN,NaN,NaN,NaN
4,E-0002036,CN(C)CCCOc1ccc2nc(-c3ccc(-c4nc5ccc(OCCCN(C)C)c...,NaN,NaN,NaN,2.212188,NaN,NaN,NaN,NaN,0.005266


### The nine endpoints

| column | what it measures | why a chemist cares |
|---|---|---|
| `LogD` | lipophilicity at pH 7.4 | partition between organic solvent and aqueous |
| `LogS` | kinetic solubility (KSOL) | solubility in aqueous buffer (affects dosage) |
| `Log_HLM_CLint` | human liver microsome clearance | how fast humans destroy it |
| `Log_MLM_CLint` | mouse liver microsome clearance | how fast the mouse model destroys it |
| `Log_Caco_Papp_AB` | passive permeability | can it cross the gut wall |
| `Log_Caco_ER` | efflux ratio | is it being actively pumped back out |
| `Log_Mouse_PPB` | plasma protein binding | how much drug binds plasma/is not free |
| `Log_Mouse_BPB` | brain protein binding | how much drug binds in brain/is not free |
| `Log_Mouse_MPB` | muscle protein binding | how much drug binds in muscle/is not free |

Notice they are *not independent*. Lipophilicity drives permeability and
protein binding.

In [4]:
counts = train[common.ENDPOINTS].notna().sum().sort_values(ascending=False)
pct = (100 * counts / len(train)).round(1)
import pandas as pd
pd.DataFrame({"measured": counts, "% of molecules": pct})

,measured,% of molecules
LogS,5128,96.3
LogD,5039,94.6
Log_MLM_CLint,4522,84.9
Log_HLM_CLint,3759,70.6
Log_Caco_ER,2161,40.6
Log_Caco_Papp_AB,2157,40.5
Log_Mouse_PPB,1302,24.4
Log_Mouse_BPB,975,18.3
Log_Mouse_MPB,222,4.2


That sparsity is not a defect in the dataset; it is what
real project data looks like.

**What strategies can you use to account for the uneven presence/absence of labels?**

Brainstorm! This is one of the key choices you'll have to make for your modeling

> `An idea for addressing uneven labeling`

---
## 2. How you are scored

The leaderboard metric is **MA-RAE**: macro-averaged relative absolute error.

For one endpoint, RAE is your total absolute error divided by the total
absolute error of a model that always guesses the mean:

$$\mathrm{RAE} = \frac{\sum |y - \hat{y}|}{\sum |y - \bar{y}|}$$

which gives it a very convenient reading:

| RAE | meaning |
|---|---|
| 1.0 | as good as guessing the average **of the set you are scored on** |
| 0.5 | half the error of guessing (roughly the winning entry) |
| >1.0 | actively worse than guessing |

RAE is computed for each endpoint, and then MA-RAE is then the plain average of these values. **Why might we want to average the RAE of each endpoint as opposed to averaging the error across all endpoints together?**

For reference, from the real challenge:

| entry | MA-RAE |
|---|---|
| *pebble* (Inductive Bio) &mdash; winner | **0.511** |
| median finisher of 103 ranked entries | ~0.68 |
| guessing the mean | 1.00 |

Additionally, we will have individual leaderboards for each endpoint. These will be scored by MAE or Mean Absolute Error.

### &#9654;&#65039; Predict first

**You are about to build a model that predicts the TRAINING mean for every molecule.**

**If it gets scored on the training data:** Will it score MA-RAE of exactly 1.0, a bit under, or a bit over?

**If it gets scored on unseen data:** Will it score MA-RAE of exactly 1.0, a bit under, or a bit over?

*Read the definition of RAE again and ask: whose mean is in the numerator, and whose is in the denominator?*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

---
## 3. The null model


In [5]:
pred = common.predict_mean_baseline(train, test)
pred.head()

,Molecule Name,LogD,LogS,Log_HLM_CLint,Log_MLM_CLint,Log_Caco_Papp_AB,Log_Caco_ER,Log_Mouse_PPB,Log_Mouse_BPB,Log_Mouse_MPB
0,E-0020101,2.111689,-4.165382,1.254441,2.228533,-5.046666,0.507704,0.972759,0.693049,0.744712
1,E-0020102,2.111689,-4.165382,1.254441,2.228533,-5.046666,0.507704,0.972759,0.693049,0.744712
2,E-0020103,2.111689,-4.165382,1.254441,2.228533,-5.046666,0.507704,0.972759,0.693049,0.744712
3,E-0020104,2.111689,-4.165382,1.254441,2.228533,-5.046666,0.507704,0.972759,0.693049,0.744712
4,E-0020105,2.111689,-4.165382,1.254441,2.228533,-5.046666,0.507704,0.972759,0.693049,0.744712


In [8]:
# Score it against the TRAINING data

fold, _ = common.load_split(train)
tr, va = train[fold == "train"], train[fold == "val"]

tr_pred = common.predict_mean_baseline(tr, tr)
common.evaluate(tr, tr_pred).round(3)

no saved split found -- using the default temporal split. Run 02_validation.ipynb to choose your own.


,n,MAE,RAE,R2,Spearman,Kendall
endpoint,,,,,,
LogD,3975,0.924,1.0,0.0,NaN,NaN
LogS,4068,0.582,1.0,0.0,NaN,NaN
Log_HLM_CLint,3600,0.498,1.0,-0.0,NaN,NaN
Log_MLM_CLint,3680,0.547,1.0,0.0,NaN,NaN
Log_Caco_Papp_AB,1936,0.355,1.0,0.0,NaN,NaN
Log_Caco_ER,1940,0.216,1.0,0.0,NaN,NaN
Log_Mouse_PPB,1098,0.398,1.0,0.0,NaN,NaN
Log_Mouse_BPB,773,0.367,1.0,0.0,NaN,NaN
Log_Mouse_MPB,111,0.295,1.0,0.0,NaN,NaN


In [9]:
# Score it against a validation set: here, a held-out slice of the TRAINING data
# (since we cannot score against the blinded test labels.)

fold, _ = common.load_split(train)
tr, va = train[fold == "train"], train[fold == "val"]

va_pred = common.predict_mean_baseline(tr, va)
common.evaluate(va, va_pred).round(3)

no saved split found -- using the default temporal split. Run 02_validation.ipynb to choose your own.


,n,MAE,RAE,R2,Spearman,Kendall
endpoint,,,,,,
LogD,1064,1.027,1.083,-0.234,NaN,NaN
LogS,1060,0.669,1.033,-0.009,NaN,NaN
Log_HLM_CLint,159,0.514,0.990,-0.019,NaN,NaN
Log_MLM_CLint,842,0.940,1.311,-0.872,NaN,NaN
Log_Caco_Papp_AB,221,0.425,1.324,-0.681,NaN,NaN
Log_Caco_ER,221,0.409,1.084,-0.574,NaN,NaN
Log_Mouse_PPB,204,0.337,1.040,-0.079,NaN,NaN
Log_Mouse_BPB,202,0.395,1.413,-0.558,NaN,NaN
Log_Mouse_MPB,111,0.422,1.667,-1.264,NaN,NaN


### Why is that not 1.0?

Look again at the definition:

$$\mathrm{RAE} = \frac{\sum |y - \hat{y}|}{\sum |y - \bar{y}|}$$

The numerator uses your prediction, which is the mean of the **training**
compounds. The denominator uses the mean of the **validation** compounds. Those
are the same number only if the two sets have the same distribution.

- Score a model **in-sample** and you get exactly 1.000, always.
- Score it on a **random** holdout and you get about 1.00, because a random
  holdout has the same distribution by construction.
- Score it on a **temporal** holdout &mdash; which is what just happened, and
  what the real test set is &mdash; and you get something above 1.0.

**The excess over 1.0 is a measurement.** It tells you how far the chemistry
moved between the early compounds and the late ones. Roughly:

| null-model RAE | mean shift between train and val |
|---|---|
| 1.00 | none |
| 1.11 | ~0.5 standard deviations |
| 1.32 | ~0.75 sd |
| 1.51 | ~1.0 sd |

So an endpoint where the null model scores 1.4 is one where **the project team
changed the molecules** over the campaign. Which is the whole point of a
lead-optimisation programme: they were fixing things.

**Based on this, which endpoints do you think your model will struggle with the most?**

---
## 4. Submit

This one is **free** &mdash; it does not count against your six submissions for
the day. Everything after this does.

Note the prediction you are committing to below. Your temporal validation
number is the best guess you have, but the real test set is a *further* step
forward in time than your holdout, so the drift there is probably larger. Do
you want to nudge your estimate up?

In [15]:
null_estimate = common.evaluate(va, va_pred)["RAE"].mean()
print(f"null model on your temporal holdout: MA-RAE = {null_estimate:.3f}")

null model on your temporal holdout: MA-RAE = 1.216


In [16]:
common.submit(
    pred,
    expected_ma_rae=float(null_estimate),   # adjust if you think the drift is worse
    why="null model: predict the training mean for everything",
    warmup=True,
)

submitted to the shared folder as test__20260728T173018Z__warmup
warm-up submission -- this one is free, your budget is untouched.


How close was your estimate to what the leaderboard gave you?

### About that submission budget

You get **six** real submissions for the whole day. This means that you have to know whether your model is good before you find out whether your model is good.

Participants in the real challenge could submit freely, and many of them
overfit the public leaderboard so hard that their rank collapsed when the
final scores came out on the full test set. One entry fell 40 places. Another
climbed 50. You will see the same thing happen at the reveal this afternoon.